<a href="https://colab.research.google.com/github/Themostamazing-bot/Friedman-numerical-implementation/blob/main/friedmann_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Friedmann Equations, ΛCDM Model, and Cosmological Distances
### Numerical Implementation — Corrected Final Version
**Author:** Tsotetsi Lehlohonolo (4351164) · University of the Western Cape  
**Supervisor:** Prof. Roy Maartens · **Co-supervisor:** Dr. Yolanda Dube  
**Python ≥ 3.10 | NumPy ≥ 1.24 | SciPy ≥ 1.11 | Matplotlib ≥ 3.7**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.integrate import solve_ivp, quad
from scipy.special import hyp2f1
import warnings, sys
warnings.filterwarnings('ignore')

# ── Physical constants ────────────────────────────────────────────────────
H0_kms   = 67.4          # Hubble constant [km s⁻¹ Mpc⁻¹]  (Planck 2018)
H0_Gyr   = H0_kms / 977.792  # Hubble constant [Gyr⁻¹]
c_kms    = 2.998e5       # speed of light [km/s]
DH       = c_kms / H0_kms   # Hubble distance [Mpc]

# ── Fiducial Planck 2018 parameters ───────────────────────────────────────
OM0  = 0.30   # Ω_{m,0}
OL0  = 0.70   # Ω_{Λ,0}
OR0  = 0.0    # Ω_{r,0} (neglected for ΛCDM runs)

# ── Wong (2011) colour-blind safe palette ─────────────────────────────────
C_BLUE   = '#0072B2'
C_ORANGE = '#E69F00'
C_GREEN  = '#009E73'
C_VERMIL = '#D55E00'
C_PURPLE = '#CC79A7'
C_SKY    = '#56B4E9'

# ── Global matplotlib style ───────────────────────────────────────────────
plt.rcParams.update({
    'font.size': 13, 'axes.labelsize': 13, 'axes.titlesize': 13,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'legend.fontsize': 11, 'figure.dpi': 130,
    'lines.linewidth': 2.0, 'axes.grid': True,
    'grid.alpha': 0.3, 'grid.linestyle': '--',
})

print(f'NumPy {np.__version__}, SciPy {__import__("scipy").__version__}, '
      f'Matplotlib {__import__("matplotlib").__version__}')
print(f'H₀ = {H0_kms} km/s/Mpc = {H0_Gyr:.6f} Gyr⁻¹')
print(f'Hubble distance D_H = c/H₀ = {DH:.1f} Mpc')


In [ ]:
# ── Dimensionless squared Hubble rate ────────────────────────────────────
def E2_a(a, Om=OM0, OL=OL0, Or=0.0):
    """E²(a) = H²(a)/H₀²  (scale-factor form)."""
    Ok = 1.0 - Om - OL - Or
    return Or*a**(-4) + Om*a**(-3) + Ok*a**(-2) + OL

def E2_z(z, Om=OM0, OL=OL0, Or=0.0):
    """E²(z) = H²(z)/H₀²  (redshift form)."""
    return E2_a(1.0/(1.0+z), Om=Om, OL=OL, Or=Or)

# Sanity check at a=1 (today)
assert abs(E2_a(1.0) - 1.0) < 1e-12, 'E²(a=1) must equal 1'
print('E²(a=1) =', E2_a(1.0), '  ✓')


In [ ]:
def rhs(t, y, Om, OL, Or=0.0):
    """First-order IVP: da/dt = H₀ a E(a)."""
    a = y[0]
    if a <= 0:
        return [0.0]
    return [H0_Gyr * a * np.sqrt(max(E2_a(a, Om, OL, Or), 0.0))]

def solve_a(Om=OM0, OL=OL0, Or=0.0, t_max=30.0, dense=True):
    """
    Integrate a(t) from near the Big Bang to t_max [Gyr].
    Returns (t_array, a_array) using DOP853 with tight tolerances.
    """
    a_init = 1e-6
    if Or > 0:
        t_init = a_init / (H0_Gyr * np.sqrt(Or)) / 2.0
    elif Om > 0:
        t_init = (2.0/3.0) * a_init**1.5 / (H0_Gyr * np.sqrt(Om))
    else:
        t_init = 1e-6
    sol = solve_ivp(
        rhs, [t_init, t_max], [a_init],
        args=(Om, OL, Or),
        method='DOP853', rtol=1e-10, atol=1e-12,
        max_step=0.05, dense_output=dense
    )
    return sol.t, sol.y[0], sol

def find_t0(Om=OM0, OL=OL0, Or=0.0):
    """Age of the universe [Gyr]: time when a=1."""
    from scipy.optimize import brentq
    t_arr, a_arr, sol = solve_a(Om, OL, Or, t_max=25.0)
    if a_arr[-1] < 1.0:
        return None
    f = lambda t: sol.sol(t)[0] - 1.0
    return brentq(f, t_arr[0], t_arr[-1])

t0_LCDM   = find_t0(OM0, OL0)
t0_matter = find_t0(1.0, 0.0)
print(f'ΛCDM age     t₀ = {t0_LCDM:.3f} Gyr   (expected ≈ 14.0 Gyr)')
print(f'Matter-only  t₀ = {t0_matter:.3f} Gyr  (expected ≈  9.7 Gyr)')


In [ ]:
# ── Analytic solutions ────────────────────────────────────────────────────
t_gyr = np.linspace(0.01, 25.0, 5000)

# Matter-only: a ∝ t^{2/3},  normalised so a(t₀)=1
a_matter_analytic = (t_gyr / t0_matter)**(2.0/3.0)

# Radiation-only: a ∝ t^{1/2}, normalised so a=1 at 9.7 Gyr for display
a_rad_analytic = (t_gyr / t0_matter)**0.5

# ΛCDM analytic: a = (Ω_m/Ω_Λ)^{1/3} sinh^{2/3}[3/2 √Ω_Λ H₀ t]
sinh_arg = 1.5 * np.sqrt(OL0) * H0_Gyr * t_gyr
a_LCDM_analytic = (OM0/OL0)**(1.0/3.0) * np.sinh(sinh_arg)**(2.0/3.0)

# De Sitter (Ω_Λ=1): a = exp[H_Λ (t−t₀)]
H_dS = H0_Gyr * np.sqrt(OL0)
a_dS = np.exp(H_dS * (t_gyr - t0_LCDM))

# ── Key derived values ────────────────────────────────────────────────────
q0    = OM0/2.0 - OL0                      # deceleration parameter
j0    = 1.0                                # jerk (exact for flat ΛCDM)
s0    = 1.0 - 9.0*OM0/2.0                 # snap parameter

# Transition redshift z_acc: ρ_Λ = ρ_m/2 → OL*(1+z)^0 = OM*(1+z)^3/2
z_acc = (2.0*OL0/OM0)**(1.0/3.0) - 1.0

# Matter–Λ equality: Ω_m(1+z)^3 = Ω_Λ
z_eq  = (OL0/OM0)**(1.0/3.0) - 1.0

print(f'Deceleration parameter  q₀ = {q0:.4f}  (expected −0.55)')
print(f'Jerk parameter          j₀ = {j0:.4f}  (expected  1.00, exact)')
print(f'Snap parameter          s₀ = {s0:.4f}  (expected −0.35)')
print(f'Acceleration transition z_acc = {z_acc:.4f}  (expected ≈ 0.67)')
print(f'Matter–Λ equality       z_eq  = {z_eq:.4f}  (expected ≈ 0.33)')


In [ ]:
# ── Comoving (transverse) distance — general curvature ───────────────────
def DM(z, Om=OM0, OL=OL0, Or=0.0):
    """
    Transverse comoving distance D_M(z) [Mpc].
    For flat models D_M = comoving line-of-sight distance.
    For curved models curvature corrections are applied via:
      Open  (Ok>0): D_M = D_H/√Ok · sinh(√Ok · χ̃)
      Closed(Ok<0): D_M = D_H/√|Ok| · sin(√|Ok| · χ̃)
    where χ̃ = H₀χ/(c) and χ is the line-of-sight integral.
    """
    Ok = 1.0 - Om - OL - Or
    E  = lambda zp: np.sqrt(E2_z(zp, Om, OL, Or))
    I, _ = quad(lambda zp: 1.0/E(zp), 0, z, limit=500,
                epsabs=1e-10, epsrel=1e-10)
    chi = DH * I           # line-of-sight comoving distance [Mpc]
    if abs(Ok) < 1e-6:     # flat
        return chi
    elif Ok > 0:           # open
        sqOk = np.sqrt(Ok)
        return (DH / sqOk) * np.sinh(sqOk * I)
    else:                  # closed
        sqOk = np.sqrt(-Ok)
        return (DH / sqOk) * np.sin(sqOk * I)

def dL(z, Om=OM0, OL=OL0, Or=0.0):
    """Luminosity distance [Mpc]: d_L = (1+z) D_M."""
    return (1.0+z) * DM(z, Om, OL, Or)

def dA(z, Om=OM0, OL=OL0, Or=0.0):
    """Angular diameter distance [Mpc]: d_A = D_M/(1+z)."""
    return DM(z, Om, OL, Or) / (1.0+z)

def mu(z, Om=OM0, OL=OL0, Or=0.0):
    """Distance modulus μ = 5 log₁₀(d_L / 10 pc)."""
    return 5.0 * np.log10(dL(z, Om, OL, Or) * 1e6 / 10.0)

def dL_taylor(z, q0=q0, j0=j0):
    """
    Cosmographic Taylor expansion of d_L to O(z²):
      d_L = (c/H₀) z [1 + (1-q₀)/2 z + O(z²)]
    The O(z³) coefficient from Visser (2004):
      coeff3 = (1/6)[−1 − q₀ + q₀² + 3q₀² − j₀ + 2Ω_k]
    For flat ΛCDM, 2Ω_k=0 and j₀=1.
    """
    c1 = 1.0
    c2 = (1.0 - q0) / 2.0
    c3 = (-1.0 - 3.0*q0 + j0 + q0**2) / 6.0  # Visser (2004) Eq. 3.3, flat
    return (c_kms/H0_kms) * z * (c1 + c2*z + c3*z**2)

# ── Verification: χ(z=1) ─────────────────────────────────────────────────
chi_z1 = DM(1.0)
dL_z1  = dL(1.0)
dA_z1  = dA(1.0)
print(f'χ(z=1)  = {chi_z1:.1f} Mpc   (expected 3431.3 Mpc)')
print(f'd_L(z=1)= {dL_z1:.1f} Mpc')
print(f'd_A(z=1)= {dA_z1:.1f} Mpc')
print(f'Etherington check: d_L/d_A/(1+z)^2 = {dL_z1/dA_z1/(2.0)**2:.8f}  (must be 1)')

# ── Particle horizon (ΛCDM) ───────────────────────────────────────────────
# d_H = c ∫₀^{t₀} dt/a(t) = (c/H₀) ∫₀^∞ dz/E(z)
I_horizon, _ = quad(lambda z: 1.0/np.sqrt(E2_z(z)), 0, np.inf,
                     limit=1000, epsabs=0, epsrel=1e-8)
d_horizon_Mpc = DH * I_horizon
d_horizon_Gly = d_horizon_Mpc * 3.2616e-3  # 1 Mpc = 3.2616 Mly
print(f'\nParticle horizon d_H(ΛCDM) = {d_horizon_Mpc:.0f} Mpc '
      f'= {d_horizon_Mpc/1e3:.1f} Gpc  (expected ≈ 14.7 Gpc)')
print(f'                           = {d_horizon_Gly:.1f} Gly  (expected ≈ 47.9 Gly)')
print(f'Matter-only: d_H = 3 c t₀ = {3*c_kms/977.792*t0_matter:.0f} Mpc')


In [ ]:
# ── Cosmographic parameters from a(t) ────────────────────────────────────
t_arr, a_arr, sol_LCDM = solve_a(OM0, OL0, t_max=25.0)

# Evaluate at t₀
a0_val  = sol_LCDM.sol(t0_LCDM)[0]

# Derivatives via ODE rhs (the Friedmann equation gives ȧ exactly)
def adot(t, Om=OM0, OL=OL0):
    a_t = sol_LCDM.sol(t)[0]
    return H0_Gyr * a_t * np.sqrt(E2_a(a_t, Om, OL))

h  = 1e-4  # finite difference step [Gyr]
a1 = sol_LCDM.sol(t0_LCDM)[0]
ad1 = adot(t0_LCDM)

# H₀ from numerical solution
H0_num = ad1 / a1

# q₀ from acceleration equation: q = -(ä a)/ȧ² = Ω_m/2 - Ω_Λ
q0_check = OM0/2.0 - OL0

# j₀ = 1 exactly for flat ΛCDM
j0_check = 1.0 + OM0*(1.5*OM0 - OL0)/(OM0 + OL0)  # simplified exact
# Actually j₀=1 exactly for any flat ΛCDM (Visser 2004)
j0_check = 1.0

print(f'Numerical H₀  = {H0_num*977.792:.3f} km/s/Mpc  (input: {H0_kms})')
print(f'Analytic  q₀  = {q0_check:.4f}             (expected −0.55)')
print(f'Analytic  j₀  = {j0_check:.4f}             (expected 1.00 exactly)')
print(f'Analytic  s₀  = {s0:.4f}             (expected −0.35)')


In [ ]:
# ── Fig 1 : Expansion histories ──────────────────────────────────────────
t_max_plot = 25.0
t_arr_M, a_arr_M, _ = solve_a(1.0, 0.0, t_max=t_max_plot)
t_arr_R, a_arr_R, _ = solve_a(0.0, 0.0, Or=1.0, t_max=t_max_plot)
t_arr_L, a_arr_L, _ = solve_a(OM0, OL0, t_max=t_max_plot)
t_arr_dS, a_arr_dS, _ = solve_a(0.0, 1.0, t_max=t_max_plot)

# Analytic overlays on common time grid
t_an = np.linspace(0.05, t_max_plot, 2000)
a_m_an  = (t_an/t0_matter)**(2.0/3.0)
a_r_an  = (t_an/(t0_matter/np.sqrt(1.0)))**0.5   # normalised differently
sinh_an = 1.5*np.sqrt(OL0)*H0_Gyr*t_an
a_L_an  = (OM0/OL0)**(1.0/3.0)*np.sinh(sinh_an)**(2.0/3.0)
a_dS_an = np.exp(H0_Gyr*np.sqrt(OL0)*(t_an - t0_LCDM))

fig, ax = plt.subplots(figsize=(8, 5))

# Numerical (solid thick lines)
ax.plot(t_arr_L, a_arr_L, color=C_BLUE,   lw=2.5, label=r'$\Lambda$CDM ($\Omega_m=0.3, \Omega_\Lambda=0.7$) — numerical')
ax.plot(t_arr_M, a_arr_M, color=C_ORANGE, lw=2.5, label=r'Matter-only ($\Omega_m=1$) — numerical')
ax.plot(t_arr_R, a_arr_R, color=C_GREEN,  lw=2.5, label=r'Radiation-only — numerical')
ax.plot(t_arr_dS, a_arr_dS, color=C_PURPLE, lw=2.5, label=r'De Sitter ($\Omega_\Lambda=1$) — numerical')

# Analytic (dashed thin, plotted on top)
ax.plot(t_an, a_L_an,  color=C_BLUE,   lw=1.2, ls='--', label='ΛCDM analytic')
ax.plot(t_an, a_m_an,  color=C_ORANGE, lw=1.2, ls='--', label='Matter analytic')
ax.plot(t_an, a_dS_an, color=C_PURPLE, lw=1.2, ls='--', label='De Sitter analytic')

# Mark today
ax.axvline(t0_LCDM, color='grey', lw=1.0, ls=':', alpha=0.7)
ax.text(t0_LCDM+0.3, 0.15, f'$t_0={t0_LCDM:.1f}$ Gyr', fontsize=10, color='grey')
ax.axhline(1.0, color='grey', lw=0.8, ls=':', alpha=0.5)

ax.set_xlabel('Time $t$ [Gyr]')
ax.set_ylabel('Scale factor $a(t)$')
ax.set_xlim(0, t_max_plot)
ax.set_ylim(0, 4.0)
ax.legend(fontsize=9, loc='upper left', ncol=1)
plt.tight_layout()
plt.savefig('fig1_expansion_histories.pdf', bbox_inches='tight')
plt.savefig('fig1_expansion_histories.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 1 saved.')


In [ ]:
# ── Fig 2 : Hubble parameter H(z) ────────────────────────────────────────
# Physical note: at z>0 we look back to when the universe was denser.
# For ΛCDM with Ω_m=0.3, Ω_Λ=0.7:
#   E²(z) = 0.3(1+z)³ + 0.7
# At every z>0 the ΛCDM value of H is LESS than the matter-only value
# because the cosmological constant does not contribute to H at high z
# while matter-only has all density as matter which scales as (1+z)³.
# Wait — let's check. For matter-only OM=1:
#   E²_m(z) = (1+z)³
# For ΛCDM OM=0.3, OL=0.7:
#   E²_L(z) = 0.3(1+z)³ + 0.7
# At z=0: E²_m=1=E²_L ✓ (both normalised to 1)
# At z=1: E²_m=8, E²_L=0.3*8+0.7=3.1  →  H_LCDM < H_matter
# So: ΛCDM lies BELOW matter-only at all z>0.

z_arr = np.linspace(0, 3, 500)

H_LCDM   = H0_kms * np.array([np.sqrt(E2_z(z, OM0, OL0)) for z in z_arr])
H_matter = H0_kms * np.array([np.sqrt(E2_z(z, 1.0, 0.0)) for z in z_arr])
H_dS     = H0_kms * np.array([np.sqrt(E2_z(z, 0.0, 1.0)) for z in z_arr])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(z_arr, H_LCDM,   color=C_BLUE,   label=r'$\Lambda$CDM ($\Omega_m=0.3,\,\Omega_\Lambda=0.7$)')
ax.plot(z_arr, H_matter, color=C_ORANGE,  label=r'Matter-only ($\Omega_m=1$)')
ax.plot(z_arr, H_dS,     color=C_PURPLE,  label=r'De Sitter ($\Omega_\Lambda=1$)')

ax.axvline(z_acc, color='grey', lw=1.0, ls=':', alpha=0.7)
ax.text(z_acc+0.05, 50, f'$z_{{\\rm acc}}={z_acc:.2f}$', fontsize=10, color='grey')

ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'$H(z)$ [km s$^{-1}$ Mpc$^{-1}$]')
ax.legend()
plt.tight_layout()
plt.savefig('fig2_hubble.pdf', bbox_inches='tight')
plt.savefig('fig2_hubble.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 2 saved.')
print(f'H_LCDM(z=1) = {H0_kms*np.sqrt(E2_z(1.0,OM0,OL0)):.1f} km/s/Mpc')
print(f'H_matter(z=1) = {H0_kms*np.sqrt(E2_z(1.0,1.0,0.0)):.1f} km/s/Mpc')
print('ΛCDM lies BELOW matter-only at z>0, as expected.')


In [ ]:
# ── Fig 3 : Distance modulus μ(z) ────────────────────────────────────────
# Three models:
#   (a) Flat ΛCDM:         Ω_m=0.3, Ω_Λ=0.7, Ω_k=0
#   (b) Flat matter-only:  Ω_m=1.0, Ω_Λ=0.0, Ω_k=0
#   (c) Open matter-only:  Ω_m=0.3, Ω_Λ=0.0, Ω_k=0.7  ← non-flat!
# For model (c) the transverse comoving distance uses the correct
# open-universe formula (Peebles 1993, Hogg 1999):
#   D_M = D_H/√Ω_k · sinh(√Ω_k · I)  where I=∫dz'/E(z')
# This is handled by the DM() function with curvature correction.

z_mu  = np.linspace(0.01, 2.5, 400)

mu_LCDM  = np.array([mu(z, OM0,  OL0,  0.0) for z in z_mu])
mu_flat  = np.array([mu(z, 1.0,  0.0,  0.0) for z in z_mu])
mu_open  = np.array([mu(z, 0.30, 0.0,  0.0) for z in z_mu])  # DM uses sinh correction

# Verify open model uses correct formula
Ok_open = 1.0 - 0.3 - 0.0
print(f'Open model: Ω_k = {Ok_open:.1f}  (positive → open geometry)')
print(f'DM(z=1, open) = {DM(1.0, 0.3, 0.0):.1f} Mpc')
print(f'DM(z=1, ΛCDM) = {DM(1.0, OM0, OL0):.1f} Mpc')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(z_mu, mu_LCDM, color=C_BLUE,   label=r'$\Lambda$CDM ($\Omega_m=0.3,\,\Omega_\Lambda=0.7$)')
ax.plot(z_mu, mu_flat, color=C_ORANGE,  label=r'Flat matter-only ($\Omega_m=1$)')
ax.plot(z_mu, mu_open, color=C_GREEN,   label=r'Open matter ($\Omega_m=0.3,\,\Omega_\Lambda=0$, $\Omega_k=0.7$)')

ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'Distance modulus $\mu$ [mag]')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('fig3_distance_modulus.pdf', bbox_inches='tight')
plt.savefig('fig3_distance_modulus.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 3 saved.')


In [ ]:
# ── Figs 4–7 : Parameter sweep over Ω_{m,0} ─────────────────────────────
Om_vals = np.arange(0.1, 1.05, 0.1)  # includes 1.0
# Use colour-blind safe sequential palette (Blues)
cmap  = plt.cm.viridis
colors_sweep = [cmap(i/(len(Om_vals)-1)) for i in range(len(Om_vals))]

t_arr_all, a_arr_all = [], []
t0_vals, q0_vals, zacc_vals = [], [], []

for Om in Om_vals:
    OL = 1.0 - Om   # flat universe
    t_a, a_a, _ = solve_a(Om, OL, t_max=30.0)
    t_arr_all.append(t_a)
    a_arr_all.append(a_a)
    t0v = find_t0(Om, OL)
    t0_vals.append(t0v if t0v else np.nan)
    q0_vals.append(Om/2.0 - OL)
    if OL > 0:
        zacc_vals.append((2.0*OL/Om)**(1.0/3.0) - 1.0)
    else:
        zacc_vals.append(np.nan)

# ── Fig 4 : a(t) for each Ω_m ───────────────────────────────────────────
fig4, ax4 = plt.subplots(figsize=(8.5, 5.5))
for i, Om in enumerate(Om_vals):
    ax4.plot(t_arr_all[i], a_arr_all[i], color=colors_sweep[i],
             label=f'$\\Omega_m={Om:.1f}$')

# Annotate t₀ values at a=1: avoid overlaps by staggering y-offsets
y_offsets = [0.08, -0.12, 0.08, -0.12, 0.08, -0.12, 0.08, -0.12, 0.08, -0.12]
for i, (Om, t0v) in enumerate(zip(Om_vals, t0_vals)):
    if not np.isnan(t0v):
        ax4.annotate(f'{t0v:.1f}', xy=(t0v, 1.0),
                     xytext=(t0v + 0.2, 1.0 + y_offsets[i % len(y_offsets)]),
                     fontsize=7.5, color=colors_sweep[i],
                     arrowprops=dict(arrowstyle='-', color=colors_sweep[i], lw=0.5))

ax4.axhline(1.0, color='grey', lw=0.8, ls=':', alpha=0.5)
ax4.set_xlabel('Time $t$ [Gyr]')
ax4.set_ylabel('Scale factor $a(t)$')
ax4.set_xlim(0, 22)
ax4.set_ylim(0, 3.5)
# Colour bar instead of cluttered legend
sm = plt.cm.ScalarMappable(cmap=cmap,
     norm=plt.Normalize(vmin=Om_vals[0], vmax=Om_vals[-1]))
sm.set_array([])
cbar = fig4.colorbar(sm, ax=ax4, pad=0.02)
cbar.set_label(r'$\Omega_{m,0}$', fontsize=12)
plt.tight_layout()
plt.savefig('fig4_sweep_at.pdf', bbox_inches='tight')
plt.savefig('fig4_sweep_at.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 4 saved.')

# ── Fig 5 : q₀(Ω_m) ──────────────────────────────────────────────────────
Om_cont = np.linspace(0.05, 1.0, 300)
q0_cont = Om_cont/2.0 - (1.0 - Om_cont)
# Boundary: q₀=0 → Ω_m = 2/3
Om_boundary = 2.0/3.0

fig5, ax5 = plt.subplots(figsize=(7, 4.5))
ax5.plot(Om_cont, q0_cont, color=C_BLUE, lw=2)
ax5.axhline(0, color='grey', lw=1.0, ls='--', alpha=0.7)
ax5.axvline(Om_boundary, color=C_VERMIL, lw=1.5, ls='--',
            label=f'$\\Omega_m = 2/3$ ($q_0=0$ boundary)')
ax5.scatter([OM0], [q0], color=C_ORANGE, zorder=5, s=60,
            label=f'Planck 2018: $\\Omega_m={OM0}$, $q_0={q0:.2f}$')
ax5.set_xlabel(r'$\Omega_{m,0}$')
ax5.set_ylabel(r'Deceleration parameter $q_0$')
ax5.legend()
ax5.fill_between(Om_cont, q0_cont, 0, where=(q0_cont < 0),
                 alpha=0.12, color=C_BLUE, label='Accelerating')
plt.tight_layout()
plt.savefig('fig5_q0_sweep.pdf', bbox_inches='tight')
plt.savefig('fig5_q0_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Fig 5 saved.  q₀=0 at Ω_m = 2/3 = {2/3:.4f}')

# ── Fig 6 : z_acc(Ω_m) ────────────────────────────────────────────────────
Om_acc_range = Om_cont[Om_cont < 1.0]  # OL=1-Om > 0 required for acceleration
zacc_cont    = (2.0*(1.0-Om_acc_range)/Om_acc_range)**(1.0/3.0) - 1.0
# Only real solutions where zacc > 0: need OL > Om/2 → Om < 2/3
mask = (zacc_cont > 0) & np.isfinite(zacc_cont)

fig6, ax6 = plt.subplots(figsize=(7, 4.5))
ax6.plot(Om_acc_range[mask], zacc_cont[mask], color=C_BLUE, lw=2)
ax6.scatter([OM0], [z_acc], color=C_ORANGE, zorder=5, s=60,
            label=f'Planck 2018: $\\Omega_m={OM0}$, $z_{{\\rm acc}}={z_acc:.2f}$')
ax6.axvline(Om_boundary, color=C_VERMIL, lw=1.5, ls='--', alpha=0.7,
            label=r'$\Omega_m = 2/3$ (no acceleration above this)')
ax6.set_xlabel(r'$\Omega_{m,0}$')
ax6.set_ylabel(r'Transition redshift $z_{\rm acc}$')
ax6.set_xlim(0.05, 0.70)
ax6.legend()
plt.tight_layout()
plt.savefig('fig6_zacc_sweep.pdf', bbox_inches='tight')
plt.savefig('fig6_zacc_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 6 saved.  (Models with Ω_m ≥ 2/3 never accelerate, excluded.)')

# ── Fig 7 : t₀(Ω_m) ──────────────────────────────────────────────────────
# Age constraint: t₀ > 12.5 Gyr (minimum stellar age estimate)
# Find critical Ω_m threshold
t_min = 13.0  # Gyr — minimum age from oldest stellar populations (Krauss & Chaboyer 2003)
Om_dense = np.linspace(0.05, 1.0, 200)
t0_dense  = []
for Om in Om_dense:
    OL = 1.0 - Om
    t0v = find_t0(Om, OL)
    t0_dense.append(t0v if t0v else np.nan)
t0_dense = np.array(t0_dense)

# Find crossing
valid = np.isfinite(t0_dense) & (t0_dense > 0)
from scipy.interpolate import interp1d
f_t0 = interp1d(Om_dense[valid], t0_dense[valid] - t_min)
from scipy.optimize import brentq
Om_cross = brentq(f_t0, 0.1, 0.9)

fig7, ax7 = plt.subplots(figsize=(7, 4.5))
ax7.plot(Om_dense[valid], t0_dense[valid], color=C_BLUE, lw=2)
ax7.axhline(t_min, color=C_VERMIL, lw=1.5, ls='--',
            label=f'Minimum stellar age ({t_min} Gyr)')
ax7.axvline(Om_cross, color=C_GREEN, lw=1.5, ls=':',
            label=f'Age constraint: $\\Omega_m \\lesssim {Om_cross:.2f}$')
ax7.scatter([OM0], [t0_LCDM], color=C_ORANGE, zorder=5, s=60,
            label=f'Planck 2018: $\\Omega_m={OM0}$, $t_0={t0_LCDM:.1f}$ Gyr')
ax7.set_xlabel(r'$\Omega_{m,0}$')
ax7.set_ylabel(r'Age $t_0$ [Gyr]')
ax7.legend()
plt.tight_layout()
plt.savefig('fig7_t0_sweep.pdf', bbox_inches='tight')
plt.savefig('fig7_t0_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Fig 7 saved.  Age constraint: Ω_m ≲ {Om_cross:.2f}  (expected ≈ 0.39)')


In [ ]:
# ── Fig 8 : Cosmographic Taylor expansion vs exact d_L ───────────────────
# The O(z²) expansion: d_L ≈ (c/H₀)z[1 + (1-q₀)/2·z + c₃z²]
# breaks down at high z. We find the 1% threshold redshift.
# NB: range must extend to z≈1 to capture the actual breakdown.

z_cg   = np.linspace(0.001, 1.5, 600)

dL_exact = np.array([dL(z) for z in z_cg])
dL_tay   = np.array([dL_taylor(z) for z in z_cg])

rel_err  = np.abs(dL_tay - dL_exact) / dL_exact * 100.0  # percent

# Find 1% breakdown
idx_break = np.argmax(rel_err > 1.0)
z_break   = z_cg[idx_break] if rel_err[idx_break] > 1.0 else None

print(f'Cosmographic 1% breakdown at z ≈ {z_break:.3f}  (O(z³) Visser expansion)')
print(f'Max relative error in [0,1.5]: {rel_err.max():.1f}%')

fig8, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

ax8a = axes[0]
ax8a.plot(z_cg, dL_exact/1e3, color=C_BLUE,   label=r'Exact $d_L(z)$')
ax8a.plot(z_cg, dL_tay/1e3,   color=C_ORANGE,  ls='--',
          label=r'Cosmographic $O(z^2)$ expansion')
if z_break:
    ax8a.axvline(z_break, color=C_VERMIL, lw=1.0, ls=':',
                 label=f'1% deviation at $z={z_break:.2f}$')
ax8a.set_ylabel(r'$d_L$ [Gpc]')
ax8a.legend()

ax8b = axes[1]
ax8b.plot(z_cg, rel_err, color=C_BLUE, lw=2)
ax8b.axhline(1.0, color=C_VERMIL, lw=1.5, ls='--', label='1% threshold')
if z_break:
    ax8b.axvline(z_break, color=C_VERMIL, lw=1.0, ls=':')
ax8b.set_xlabel('Redshift $z$')
ax8b.set_ylabel(r'Relative error [\%]')
ax8b.set_ylim(-0.2, min(rel_err.max()*1.1, 30))
ax8b.legend()

plt.tight_layout()
plt.savefig('fig8_cosmo_check.pdf', bbox_inches='tight')
plt.savefig('fig8_cosmo_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 8 saved.')


In [ ]:
# ── Fig 9 : Numerical precision — Friedmann constraint C(t) ─────────────
#
# The correct test:
#   C(t) ≡ H²_num(t)/H₀² − E²(a(t))
# where H_num is obtained from a NUMERICAL DERIVATIVE of a(t),
# NOT from evaluating the ODE right-hand side (which is circular).
#
# Method: central finite differences on the dense output grid.
#   adot[i] ≈ [a[i+1] − a[i-1]] / [t[i+1] − t[i-1]]
#
# If the integrator truly satisfies the Friedmann equation,
# C(t) ≈ 0 to within the tolerance × truncation error of the finite diff.

t_pc, a_pc, sol_pc = solve_a(OM0, OL0, t_max=20.0, dense=True)

# Independent precision test using the dense-output polynomial.
# Instead of finite differences on the coarse adaptive step grid
# (which gives large errors at early times where steps are huge),
# we evaluate the solver's own interpolant at t±h with h=10⁻⁴ Gyr.
# The interpolant is a 7th-order polynomial that does NOT simply
# return the ODE RHS — so d/dt(interpolant) ≠ RHS in general.
# Comparing the two is therefore a genuine (non-circular) test.
t_check = np.linspace(0.5, 18.0, 2000)  # avoid t≈0 (steep gradients)
h_fd    = 1e-4  # [Gyr] — fine enough for accurate central difference
a_plus  = sol_pc.sol(t_check + h_fd)[0]
a_mid   = sol_pc.sol(t_check)[0]
a_minus = sol_pc.sol(t_check - h_fd)[0]
adot_num = (a_plus - a_minus) / (2.0 * h_fd)  # central difference
H_num    = adot_num / a_mid   # [Gyr⁻¹]

# E²(a) from the analytic Friedmann expression
E2_num = np.array([E2_a(a) for a in a_mid])

# Constraint residual C(t) — must be ≈ 0 if the integration is faithful
C_t = (H_num / H0_Gyr)**2 - E2_num

print(f'Max |C(t)| = {np.max(np.abs(C_t)):.2e}  (must be ≪ 1)')
print(f'Mean |C(t)| = {np.mean(np.abs(C_t)):.2e}')
print('Dense-output finite-difference test — genuinely non-circular.')

fig9, axes9 = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

ax9a = axes9[0]
ax9a.plot(t_pc, a_pc, color=C_BLUE, lw=2)
ax9a.axvline(t0_LCDM, color='grey', lw=1, ls=':', alpha=0.7)
ax9a.set_ylabel('Scale factor $a(t)$')
ax9a.set_xlim(0, 20)
ax9a.text(t0_LCDM+0.2, 0.2, f'$t_0={t0_LCDM:.2f}$ Gyr', fontsize=9, color='grey')

ax9b = axes9[1]
ax9b.plot(t_check, C_t, color=C_ORANGE, lw=1.5)
ax9b.axhline(0, color='grey', lw=0.8, ls='--', alpha=0.5)
ax9b.set_xlabel('Time $t$ [Gyr] (for $t \geq 0.5$ Gyr)')
ax9b.set_ylabel(r'Constraint $C(t) = H_{\rm num}^2/H_0^2 - E^2(a)$')

# Use scientific notation on y-axis
ax9b.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
ax9b.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

plt.tight_layout()
plt.savefig('fig9_precision.pdf', bbox_inches='tight')
plt.savefig('fig9_precision.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 9 saved.')


In [ ]:
# ── Summary of key numerical results ─────────────────────────────────────
print('=' * 58)
print('  SUMMARY OF KEY RESULTS  (ΛCDM: Ω_m=0.3, Ω_Λ=0.7, H₀=67.4)')
print('=' * 58)
print(f'  Age of universe   t₀        = {t0_LCDM:.3f} Gyr      [expected 14.0]')
print(f'  Age, matter-only  t₀        = {t0_matter:.3f} Gyr      [expected  9.7]')
print(f'  Deceleration      q₀        = {q0:.4f}         [expected −0.55]')
print(f'  Jerk              j₀        = {j0:.4f}         [expected  1.00]')
print(f'  Snap              s₀        = {s0:.4f}         [expected −0.35]')
print(f'  Acc. transition   z_acc     = {z_acc:.4f}         [expected  0.67]')
print(f'  Matter–Λ equality z_eq      = {z_eq:.4f}         [expected  0.33]')
print(f'  Comoving χ(z=1)            = {DM(1.0):.1f} Mpc  [expected 3431.3]')
print(f'  Luminosity dL(z=1)         = {dL(1.0):.1f} Mpc')
print(f'  Cosmo. breakdown z_break   ≈ {z_break:.2f}           [O(z³) Visser; 1% threshold]')
print(f'  Particle horizon d_H       = {d_horizon_Mpc/1e3:.2f} Gpc   [expected 14.7]')
print(f'  Age constraint   Ω_m ≲     {Om_cross:.2f}           [expected 0.39]')
print('=' * 58)
